In [6]:
#pip install djitellopy opencv-python numpy

# Velocity based takeoff

In [7]:
from djitellopy import Tello
import cv2
import numpy as np
import time
import logging

# Suppress INFO and WARNING messages from djitellopy library
logging.getLogger('djitellopy').setLevel(logging.ERROR)


In [10]:
'''
World building
'''
#####################################
# Initialization of coordinates
#####################################
# x points straight, y points left, z points up
# Initial starting position of ego drone (at the center of the picnic blanket)
ego_x = 0
ego_y = 0
ego_z = 0

# State history: stores full state (x, y, z) at each time step
state_history = []  # List of tuples: [(x, y, z), ...]

#####################################
# Define borders
#####################################
# Define bounding box around initial starting point (200 x 200)
# Box extends 100 units in each direction from center
# BOX_SIZE = 200 # random estimate rn for the bounding box of the ODD, will test based on picnic blanket
# BOX_HALF = BOX_SIZE / 2  # 100 units
# box_x_min = ego_x - BOX_HALF
# box_x_max = ego_x + BOX_HALF
# box_y_min = ego_y - BOX_HALF
# box_y_max = ego_y + BOX_HALF

#####################################
# Pursuer position (stationary)
#####################################
pursuer_active = False

# Pursuer position (will be set after takeoff to same z-height)
pursuer_x = 45  # Fixed x position of pursuer
pursuer_y = 0   # Fixed y position of pursuer
pursuer_z = None  # Will be set to takeoff height after takeoff

# Threshold distance for collision avoidance (squared distance: x^2 + y^2)
radius = 25
pursuer_threshold = radius**2  # 50^2 = 2500 (50 cm threshold)

#####################################
# Goal tracking
#####################################
goal_index = 0
goals = [
    (20, 0),   # Goal 1: Forward
    (20, 20),  # Goal 2: Forward and left
    (20, -20),   # Goal 3: Forward and Right
    (-20, -20), # Goal 4: Backward and left
    (0, 0)   # Goal 5: Backward
]

'''
Drone building
'''
#####################################
# Initialization of motion params
#####################################
upward_initialization = 20 #20 cm is the minimum
left_movement  = 20
right_movement = 20
forward_movement = 20
backward_movement = 20

#####################################
# Initialization of drone
#####################################
tello = Tello()
#print("Connecting to Tello...")
tello.connect()
print(f"Battery level: {tello.get_battery()}%")
tello.streamon()

print("Press 't' to takeoff, 'q' to land and quit.")
drone_in_air = False

#####################################
# Definitions
#####################################
def compute_switching_filter(new_x, new_y, pursuer_x, pursuer_y):
    dist_to_pursuer_sq = (new_x - pursuer_x)**2 + (new_y - pursuer_y)**2
    boolean_filter = dist_to_pursuer_sq < pursuer_threshold
    return boolean_filter # returns true if distance to pursuer is less than threshold


'''
Takeoff
'''
#####################################
# Drone maneuverance after takeoff
#####################################
try:
    # Record initial state before loop starts
    state_history.append((ego_x, ego_y, ego_z))
    
    while True:
        frame = tello.get_frame_read().frame
        frame = cv2.resize(frame, (720, 480))
        h, w, _ = frame.shape

        # Display the frame
        cv2.imshow("Tello Tracking Feed", frame)

        key = cv2.waitKey(1) & 0xFF

        # Controls for drone
        if key == ord('t') and not drone_in_air:
            tello.takeoff()
            drone_in_air = True
            print("Drone is now airborne.")
            
            # Gradual takeoff using velocity control
            target_height = 20  # cm you want to reach (minimum safe height)
            hover_speed = 10    # cm/s (upward velocity)
            steps = int(target_height / hover_speed)
            
            for _ in range(steps):
                tello.send_rc_control(0, 0, hover_speed, 0)  # (left-right, forward-back, up-down, yaw)
                ego_z += hover_speed
                state_history.append((ego_x, ego_y, ego_z))
                time.sleep(1)  # sleep 1 second per step

            # Stop upward motion and hover
            tello.send_rc_control(0, 0, 0, 0)
            pursuer_z = ego_z
            print(f"Reached hover height: {ego_z} cm")

        


        # drone movement based on goals
        if drone_in_air and goal_index < len(goals):
            target_x, target_y = goals[goal_index]
            
            # Calculate distance to current goal
            dx = target_x - ego_x
            dy = target_y - ego_y
            distance_to_goal = np.sqrt(dx**2 + dy**2)
            
            # Threshold for considering goal reached (within 5 cm)
            goal_threshold = 5
            
            if distance_to_goal > goal_threshold:
                move_step = 20
                
                # Move in x direction first
                if abs(dx) > 0:
                    # Calculate potential new position
                    new_x = ego_x + move_step if dx > 0 else ego_x - move_step
                    new_y = ego_y  # y stays the same for x movement
                    
                    # Check if movement is safe (if pursuer is active)
                    can_move_x = True
                    if pursuer_active:
                        if compute_switching_filter(new_x, new_y, pursuer_x, pursuer_y):
                            can_move_x = False
                            print(f"CANT MOVE TOWARDS GOAL {goal_index + 1} IN X DIRECTION, PURSUER BLOCKING")
                    
                    # Execute movement if safe
                    if can_move_x:
                        if dx > 0:
                            tello.move_forward(move_step)
                        else:
                            tello.move_back(move_step)
                        ego_x = new_x
                        state_history.append((ego_x, ego_y, ego_z))
                        print(f"Moving towards goal {goal_index + 1}: ({target_x}, {target_y}), Current: ({ego_x:.1f}, {ego_y:.1f})")
                        time.sleep(0.5)

                # Then move in y direction
                if abs(dy) > 0:
                    # Calculate potential new position
                    new_x = ego_x  # x stays the same for y movement
                    new_y = ego_y + move_step if dy > 0 else ego_y - move_step
                    
                    # Check if movement is safe (if pursuer is active)
                    can_move_y = True
                    if pursuer_active:
                        if compute_switching_filter(new_x, new_y, pursuer_x, pursuer_y):
                            can_move_y = False
                            print(f"CANT MOVE TOWARDS GOAL {goal_index + 1} IN Y DIRECTION, PURSUER BLOCKING")
                    
                    # Execute movement if safe
                    if can_move_y:
                        if dy > 0:
                            tello.move_left(move_step)
                        else:
                            tello.move_right(move_step)
                        ego_y = new_y
                        state_history.append((ego_x, ego_y, ego_z))
                        print(f"Moving towards goal {goal_index + 1}: ({target_x}, {target_y}), Current: ({ego_x:.1f}, {ego_y:.1f})")
                        time.sleep(0.5)

            
            else:
                # Goal reached
                print(f"Goal {goal_index + 1} reached! ({target_x}, {target_y})")
                goal_index += 1
                time.sleep(1)
                
                if goal_index >= len(goals):
                    print("All 5 goals reached! Hovering in place.")




        # Landing control
        if key == ord('q'):
            if drone_in_air:
                tello.land()
                state_history.append((ego_x, ego_y, ego_z))  # Record final state before landing
                print("Drone has landed.")
                print(f"Final state: ({ego_x}, {ego_y}, {ego_z})")
                print(f"Total states recorded: {len(state_history)}")
            break



#####################################
# Landing the drone / finish program
#####################################
except Exception as e:
    print(f"Error occurred: {e}")
    if drone_in_air:
        print("Error occurred. Landing the drone.")
        tello.land()

except KeyboardInterrupt:
    print("Keyboard Interrupt received. Landing the drone.")
    if drone_in_air:
        tello.land()
    tello.streamoff()
    cv2.destroyAllWindows()

finally:
    tello.streamoff()
    cv2.destroyAllWindows()


Battery level: 84%
Press 't' to takeoff, 'q' to land and quit.
Drone is now airborne.
Reached hover height: 20 cm
Moving towards goal 1: (20, 0), Current: (20.0, 0.0)
Goal 1 reached! (20, 0)
Moving towards goal 2: (20, 20), Current: (20.0, 20.0)
Goal 2 reached! (20, 20)
Moving towards goal 3: (20, -20), Current: (20.0, 0.0)
Moving towards goal 3: (20, -20), Current: (20.0, -20.0)
Goal 3 reached! (20, -20)
Moving towards goal 4: (-20, -20), Current: (0.0, -20.0)
Moving towards goal 4: (-20, -20), Current: (-20.0, -20.0)
Goal 4 reached! (-20, -20)
Moving towards goal 5: (0, 0), Current: (0.0, -20.0)
Moving towards goal 5: (0, 0), Current: (0.0, 0.0)
Goal 5 reached! (0, 0)
All 5 goals reached! Hovering in place.
Keyboard Interrupt received. Landing the drone.


TelloException: Command 'land' was unsuccessful for 4 tries. Latest response:	'error'